In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("data/anime.csv")

# Display first 5 rows
print("First 5 rows:")
display(df.head())

# Display dataset shape
print("\nDataset Shape:")
print(df.shape)

# Display column names
print("\nColumn Names:")
print(df.columns.tolist())

# Display data types
print("\nData Types:")
print(df.dtypes)

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())

First 5 rows:


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266



Dataset Shape:
(12294, 7)

Column Names:
['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members']

Data Types:
anime_id      int64
name            str
genre           str
type            str
episodes        str
rating      float64
members       int64
dtype: object

Missing Values:
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

Duplicate Rows:
0


In [2]:
# Check missing values
print("Missing values before preprocessing:")
print(df.isnull().sum())

# Fill missing genres
df['genre'] = df['genre'].fillna('Unknown')

# Fill missing types
df['type'] = df['type'].fillna('Unknown')

# Fill missing ratings with the median rating
df['rating'] = df['rating'].fillna(df['rating'].median())

# Convert episodes to numeric
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

# Fill missing episode values with median
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

# Check missing values after preprocessing
print("\nMissing values after preprocessing:")
print(df.isnull().sum())

# Check duplicates
print("\nNumber of duplicate rows:", df.duplicated().sum())

Missing values before preprocessing:
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

Missing values after preprocessing:
anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

Number of duplicate rows: 0


In [3]:
# Explore numerical features
print("Numerical feature statistics:")
display(df[['rating', 'episodes', 'members']].describe())

Numerical feature statistics:


,rating,episodes,members
count,12294.000000,12294.000000,1.229400e+04
mean,6.475700,12.095412,1.807134e+04
std,1.017179,46.244062,5.482068e+04
min,1.670000,1.000000,5.000000e+00
25%,5.900000,1.000000,2.250000e+02
50%,6.570000,2.000000,1.550000e+03
75%,7.170000,12.000000,9.437000e+03
max,10.000000,1818.000000,1.013917e+06


In [4]:
# Check unique values in important categorical columns
print("Number of unique genres:", df['genre'].nunique())
print("Number of anime types:", df['type'].nunique())

print("\nAnime types:")
print(df['type'].value_counts())

Number of unique genres: 3265
Number of anime types: 7

Anime types:
type
TV         3787
OVA        3311
Movie      2348
Special    1676
ONA         659
Music       488
Unknown      25
Name: count, dtype: int64


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(token_pattern=r'[^, ]+')

# Convert genre column into TF-IDF features
genre_matrix = tfidf.fit_transform(df['genre'])

print("Genre matrix shape:", genre_matrix.shape)
print("Number of genre features:", len(tfidf.get_feature_names_out()))

In [ ]:
from sklearn.preprocessing import StandardScaler

# Select numerical features
numerical_features = df[['rating', 'episodes', 'members']]

# Standardize numerical features
scaler = StandardScaler()
numerical_matrix = scaler.fit_transform(numerical_features)

print("Numerical matrix shape:", numerical_matrix.shape)

In [ ]:
from scipy.sparse import hstack, csr_matrix

# Convert numerical features to sparse matrix
numerical_sparse = csr_matrix(numerical_matrix)

# Combine genre and numerical features
combined_features = hstack([genre_matrix, numerical_sparse])

print("Combined feature matrix shape:", combined_features.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity
cosine_sim = cosine_similarity(combined_features)

print("Cosine similarity matrix shape:", cosine_sim.shape)

In [ ]:
def recommend_anime(anime_name, num_recommendations=10):
    # Find the index of the anime
    anime_indices = df[df['name'].str.lower() == anime_name.lower()].index

    if len(anime_indices) == 0:
        print("Anime not found in the dataset.")
        return

    anime_index = anime_indices[0]

    # Get similarity scores for the selected anime
    similarity_scores = cosine_sim[anime_index]

    # Sort anime by similarity score
    similar_indices = similarity_scores.argsort()[::-1]

    # Remove the selected anime itself
    similar_indices = similar_indices[similar_indices != anime_index]

    # Select top recommendations
    top_indices = similar_indices[:num_recommendations]

    # Create recommendation DataFrame
    recommendations = df.iloc[top_indices][
        ['name', 'genre', 'type', 'episodes', 'rating', 'members']
    ].copy()

    # Add similarity score
    recommendations['Similarity Score'] = similarity_scores[top_indices]

    return recommendations.reset_index(drop=True)

In [ ]:
recommend_anime("Naruto", 10)

In [ ]:
def recommend_by_threshold(anime_name, threshold=0.5, max_recommendations=10):
    # Find the anime index
    anime_indices = df[df['name'].str.lower() == anime_name.lower()].index

    if len(anime_indices) == 0:
        print("Anime not found in the dataset.")
        return

    anime_index = anime_indices[0]

    # Get similarity scores
    similarity_scores = cosine_sim[anime_index]

    # Get indices above the threshold
    similar_indices = np.where(similarity_scores >= threshold)[0]

    # Remove the selected anime
    similar_indices = similar_indices[similar_indices != anime_index]

    # Sort by similarity score
    similar_indices = similar_indices[
        np.argsort(similarity_scores[similar_indices])[::-1]
    ]

    # Limit recommendations
    similar_indices = similar_indices[:max_recommendations]

    recommendations = df.iloc[similar_indices][
        ['name', 'genre', 'type', 'episodes', 'rating', 'members']
    ].copy()

    recommendations['Similarity Score'] = similarity_scores[similar_indices]

    return recommendations.reset_index(drop=True)

In [ ]:
print("Threshold = 0.3")
display(recommend_by_threshold("Naruto", threshold=0.3, max_recommendations=10))

In [ ]:
print("Threshold = 0.5")
display(recommend_by_threshold("Naruto", threshold=0.5, max_recommendations=10))

In [ ]:
print("Threshold = 0.7")
display(recommend_by_threshold("Naruto", threshold=0.7, max_recommendations=10))

In [ ]:
thresholds = [0.3, 0.5, 0.7]

print("Recommendation Count by Similarity Threshold")
print("-" * 50)

for threshold in thresholds:
    recommendations = recommend_by_threshold(
        "Naruto",
        threshold=threshold,
        max_recommendations=100
    )

    if recommendations is not None:
        print(
            f"Threshold {threshold}: "
            f"{len(recommendations)} recommendations"
        )

In [ ]:
print("\nAverage Similarity by Threshold")
print("-" * 50)

for threshold in thresholds:
    recommendations = recommend_by_threshold(
        "Naruto",
        threshold=threshold,
        max_recommendations=100
    )

    if recommendations is not None and len(recommendations) > 0:
        avg_score = recommendations['Similarity Score'].mean()
        print(f"Threshold {threshold}: {avg_score:.4f}")

In [ ]:
recommend_anime("One Piece", 10)

In [ ]:
df[df['name'].str.contains("One Piece", case=False, na=False)][
    ['name', 'genre', 'type']
].head(10)

## Performance Analysis

The anime recommendation system was implemented using cosine similarity. The anime genres were converted into numerical features using TF-IDF, while rating, number of episodes, and number of members were standardized before combining them with the genre features.

Cosine similarity was then calculated between the feature vectors of all anime. A recommendation function was developed that accepts an anime title and returns the most similar anime based on their similarity scores.

Different similarity thresholds were tested:

* **0.3:** Produces a larger recommendation list, but some recommendations may have relatively weaker similarity.
* **0.5:** Provides a balanced set of recommendations with moderate similarity.
* **0.7:** Produces fewer recommendations, but the recommended anime have stronger similarity to the selected anime.

### Areas for Improvement

The recommendation system can be improved in several ways:

1. Include additional anime attributes such as type and popularity.
2. Give different weights to genres, ratings, episodes, and members.
3. Use user-rating information to develop a collaborative filtering model.
4. Handle anime with missing or incomplete genre information more effectively.
5. Remove extremely rare or duplicate anime titles.
6. Use a hybrid recommendation system combining content-based and collaborative filtering techniques.

## Conclusion

The cosine similarity approach successfully provides anime recommendations based on similarities between anime features. The system demonstrates how categorical information such as genres can be transformed using TF-IDF and combined with normalized numerical features.

The similarity threshold allows the recommendation system to control the trade-off between the number and quality of recommendations. Lower thresholds provide more recommendations, while higher thresholds provide fewer but more closely related recommendations.

Overall, the project demonstrates the application of feature extraction, normalization, cosine similarity, and recommendation techniques to build a basic content-based anime recommendation system.


# Interview Questions

## 1. Can you explain the difference between user-based and item-based collaborative filtering?

**Answer:**

User-based collaborative filtering recommends items based on the preferences of users who have similar interests.

For example, if User A and User B have rated many anime similarly, then anime liked by User B can be recommended to User A.

Item-based collaborative filtering recommends items based on the similarity between items.

For example, if a user likes a particular anime, the system recommends other anime that are similar to that anime based on ratings or other features.

### Difference:

- **User-based:** Finds users with similar preferences and recommends items based on those users.
- **Item-based:** Finds items that are similar to items the user has already liked.
- User-based filtering focuses on **user-to-user similarity**.
- Item-based filtering focuses on **item-to-item similarity**.

## 2. What is collaborative filtering, and how does it work?

**Answer:**

Collaborative filtering is a recommendation technique that predicts a user's preferences by using the preferences, ratings, or behavior of multiple users.

It works by identifying relationships between users and items.

The main steps are:

1. Collect user-item interaction data such as ratings, likes, or views.
2. Identify similar users or similar items.
3. Calculate the similarity between users or items.
4. Use the similarity scores to predict preferences.
5. Recommend items that the user is likely to prefer.

There are two common types of collaborative filtering:

- **User-based collaborative filtering:** Recommendations are based on users with similar preferences.
- **Item-based collaborative filtering:** Recommendations are based on items that are similar to items the user has interacted with.

Collaborative filtering is widely used in recommendation systems such as movies, music, books, and online shopping.